In [1]:
# --- 1. INSTALLATION DES DÉPENDANCES ---
print("⏳ Installation des librairies...")
!pip install -q transformers accelerate bitsandbytes moviepy openai-whisper scipy torchaudio
!pip install -q imageio-ffmpeg

print("✅ Installation terminée !")

⏳ Installation des librairies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 16.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.9 MB/s eta 0:00:00
✅ Installation terminée !


In [2]:
# --- 2. DÉFINITION DU MOTEUR IA (Architecture Multimodale) ---
import warnings
import os
import torch
import whisper
from moviepy.editor import VideoFileClip
from transformers import BlipProcessor, BlipForConditionalGeneration, MarianMTModel, MarianTokenizer
from PIL import Image
import numpy as np

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



In [3]:
class VideoInsightAI:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"⚙️ Initialisation sur : {self.device.upper()}")

        # 1. Chargement AUDIO
        print("🎧 Chargement de Whisper (Base)...")
        self.audio_model = whisper.load_model("base").to(self.device)

        # 2. Chargement VISION
        print("👁️ Chargement de BLIP (Image Captioning)...")
        self.vision_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
        self.vision_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(self.device)

        # 3. Chargement TRADUCTION (Anglais -> Français)
        print("🌍 Chargement du Traducteur (Helsinki-NLP)...")
        self.trans_model_name = "Helsinki-NLP/opus-mt-en-fr"
        self.trans_tokenizer = MarianTokenizer.from_pretrained(self.trans_model_name)
        self.trans_model = MarianMTModel.from_pretrained(self.trans_model_name).to(self.device)

        print("✅ Moteur IA prêt !")

    def _translate_to_french(self, text):
        """Traduit de l'anglais vers le français avec découpage (Chunking)"""
        if not text: return ""

        chunk_size = 1000
        chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
        translated_parts = []

        try:
            for chunk in chunks:
                inputs = self.trans_tokenizer(chunk, return_tensors="pt", padding=True, truncation=True, max_length=512).to(self.device)
                translated = self.trans_model.generate(**inputs)
                decoded_part = self.trans_tokenizer.decode(translated[0], skip_special_tokens=True)
                translated_parts.append(decoded_part)
            return " ".join(translated_parts)
        except Exception as e:
            return f"[Erreur Traduction] {str(e)}"

    def analyze_video(self, video_path):
        """Pipeline principal"""
        print(f"\n📂 Analyse du fichier : {video_path}")

        try:
            clip = VideoFileClip(video_path)
        except Exception as e:
            return f"❌ Erreur lecture vidéo : {str(e)}"

        # Vérification audio
        has_audio = clip.audio is not None

        if has_audio:
            print("🔊 Piste audio détectée -> Module WHISPER.")
            return self._process_audio(video_path)
        else:
            print("🔇 Pas d'audio détecté -> Module VISION (BLIP).")
            return self._process_images(clip)

    def _process_audio(self, video_path):
        try:
            # Whisper transcrit ET détecte la langue
            result = self.audio_model.transcribe(video_path)
            raw_text = result["text"]
            language = result["language"] # Code langue (ex: 'fr', 'en', 'es')

            print(f"📝 Langue détectée : {language.upper()}")
            print(f"📝 Transcription brute : {raw_text[:100]}...")

            # --- LOGIQUE INTELLIGENTE ---
            # On ne traduit QUE si c'est de l'anglais ('en')
            # Si c'est du Français ('fr'), on garde le texte original pour éviter les hallucinations
            if language == 'en':
                print("🇬🇧 Anglais détecté -> Traduction en cours...")
                french_text = self._translate_to_french(raw_text)
                return f"🎙️ **Transcription (Traduit de l'Anglais)** :\n{french_text}"
            else:
                print(f"🚩 Langue '{language}' détectée -> Conservation du texte original.")
                return f"🎙️ **Transcription Originale ({language.upper()})** :\n{raw_text}"

        except Exception as e:
            return f"❌ Erreur Whisper : {str(e)}"

    def _process_images(self, clip):
        try:
            duration = clip.duration
            step = max(3, duration / 5)
            timestamps = np.arange(0, duration, step)
            descriptions = []

            print(f"📸 Extraction de {len(timestamps)} images clés...")

            for t in timestamps:
                frame = clip.get_frame(t)
                image = Image.fromarray(frame)

                # BLIP parle toujours anglais
                inputs = self.vision_processor(image, return_tensors="pt").to(self.device)
                out = self.vision_model.generate(**inputs)
                caption = self.vision_processor.decode(out[0], skip_special_tokens=True)
                descriptions.append(caption)

            full_description_en = ". ".join(descriptions)

            # Ici on traduit TOUJOURS car BLIP ne sort que de l'anglais
            french_desc = self._translate_to_french(full_description_en)
            return f"🎥 **Analyse Visuelle (Scène par scène)** :\n{french_desc}"
        except Exception as e:
            return f"❌ Erreur Vision : {str(e)}"

In [4]:
# Instanciation
engine = VideoInsightAI()

⚙️ Initialisation sur : CUDA
🎧 Chargement de Whisper (Base)...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 177MiB/s]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


👁️ Chargement de BLIP (Image Captioning)...


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

...

  warnings.warn("Recommended: pip install sacremoses.")



pytorch_model.bin:   0%|          | 0.00/301M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

✅ Moteur IA prêt !


In [5]:
# --- 3. EXÉCUTION DU TEST ---

fichier_video = "analayse_real_city.mp4"

if os.path.exists(fichier_video):
    rapport = engine.analyze_video(fichier_video)

    print("\n" + "="*40)
    print("📊 RAPPORT FINAL INTELLIGENT")
    print("="*40)
    print(rapport)
else:
    print(f"❌ Fichier introuvable : {fichier_video}")


📂 Analyse du fichier : analayse_real_city.mp4
🔊 Piste audio détectée -> Module WHISPER.
📝 Langue détectée : FR
📝 Transcription brute :  Si dans ce match-là, t'es pas capable de voir l'importance de Ganchalo Garcia. Surtout en première ...
🚩 Langue 'fr' détectée -> Conservation du texte original.

📊 RAPPORT FINAL INTELLIGENT
🎙️ **Transcription Originale (FR)** :
 Si dans ce match-là, t'es pas capable de voir l'importance de Ganchalo Garcia. Surtout en première période, je ne discute pas avec toi. Je ne discuterai pas avec toi. Honnêtement, les gars Ganchalo Garcia, aujourd'hui, il m'a confirmé que on a besoin d'un numéro 9, un point de fixation. C'est une folie. On a vu la première mi-temps comment on était libérés, comment on arrivait à faire des attaques placés, parce qu'il y avait un vrai numéro 9 dans la surface. Et je vais vous dire un truc qui va vous surprendre pour moi, on a besoin de Kylane Bapé dans cet équipe. Mais sur un côté. Aujourd'hui, j'en doute même pas que tu mets Ky